# Ensemble Affinity Analysis (On-the-fly)

This notebook computes PRP and RA summaries in memory from merged benchmark files.

No summary CSVs are created and figures are shown inline only.

In [41]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

In [42]:
# Set run_id explicitly, or keep None to use latest run folder.
ignore_models = ['smollm2-1.7b', 'gemma3-4b', 'phi3-3.8b']
run_id =None
# run_id = "20260401_082538"
# run_id = "20260406_211752"
run_id = "20260413_151017"

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"

if run_id is None:
    run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
    if not run_dirs:
        raise FileNotFoundError(f"No benchmark runs found under {bench_root}")
    run_dir = run_dirs[-1]
else:
    run_dir = bench_root / run_id

prp_path = run_dir / "merged_prp_affinities.csv"
ra_path = run_dir / "merged_ra_affinities.csv"

if not prp_path.exists() :
    raise FileNotFoundError(
        "Merged files not found. Run scripts/run_postprocessing.py first for this run_id."
    )
prp_df = pd.read_csv(prp_path)
prp_df = prp_df[~prp_df['Model_Slug'].isin(ignore_models)]

print("Run directory:", run_dir)
print("PRP rows:", len(prp_df), "| Models:", prp_df['Model_Slug'].nunique())
if ra_path.exists():
    ra_df = pd.read_csv(ra_path)
    ra_df = ra_df[~ra_df['Model_Slug'].isin(ignore_models)]
    print("RA rows:", len(ra_df), "| Models:", ra_df['Model_Slug'].nunique())

    

Run directory: /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/results/affinity_benchmark/20260413_151017
PRP rows: 3000 | Models: 5
RA rows: 22637 | Models: 5


In [43]:
# Quick coverage checks
display(prp_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())
display(ra_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())

,Model_Slug,n_abstracts
0,gemini-3-flash-preview,100
1,gemma4-31b-cloud,100
2,gpt-5.4-mini,100
3,gpt-5.4-nano,100
4,ministral-3-14b-cloud,100


,Model_Slug,n_abstracts
0,gemini-3-flash-preview,100
1,gemma4-31b-cloud,100
2,gpt-5.4-mini,100
3,gpt-5.4-nano,100
4,ministral-3-14b-cloud,100


# PRP Affinity level
Affinity level of an **abstract** to each **primary research programme** (PRP)

In [44]:
fig = px.box(
    prp_df,
    x='Abstract_Index',
    y='LLM_Affinity',
    points='all',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution by Abstract'
 )
fig.update_xaxes(range=[5.5,11.5])
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
fig.show()

In [45]:
fig = px.box(
    prp_df,
    color='Model_Slug',
    y='LLM_Affinity',
    x = 'PRP_Name',
    # points = 'all',
    title='Affinity Distribution by PRP across models',
 )
# fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [46]:
# fig = px.box(
#     prp_df,
#     x='Abstract_Index',
#     y='LLM_Affinity',
#     points='all',
#     color = 'PRP_Name',
#     title=f'PRP Affinity Distribution by Abstract'
#  )
# fig.update_xaxes(range=[0.5,10.5])
# fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.show()

In [47]:
abstracts_ids = [1,8,20, 19]
abstracts_ids = [20, 25, 34]
fig = px.box(
    prp_df[prp_df['Abstract_Index'].isin(abstracts_ids)], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    points='all',
    # hover_data='PRP_Name',
    # facet_row='PRP_Name',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    # x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract'
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [48]:
fig = px.line(
    prp_df[prp_df['Abstract_Index'].isin([1,8,20, 19, 30])], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    # points='all',
    # hover_data='PRP_Name',
    # facet_row='PRP_Name',

    facet_col='Abstract_Index',
    color = 'PRP_Name',
    x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract',
    markers = True
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

General categories:
- 1 - Scenario Reduction Network Based on Wasserstein Distance With Regularization - Planning
- 8 - Simultaneous Assessment of Multiple Aspects of Stability of Power Systems With Renewable Generation -  Stability
- 19 - Chance-Constrained Pre-Contingency Joint Self- Scheduling of Energy and Reserve in VPP - DER, Services
- 20 - Dual Grid-Forming Control With Energy Regulation Capability of MMC-HVDC System Integrating Offshore Wind Farms and Weak Grids - IBR, Stability
- 30 - Synergistic Integration of Machine Learning and Mathematical Optimization for Unit Commitment - ????

Observations:
- Gemma: fails to identify planning for Index 1
- Gemma cloud: best classification
- Llama: miscelaneous performance
- Mistral: systematically assign too high affinities
- Phi & Smollm: systematically assign too high affinities
- Qwen: miscelaneous


## PRP Summaries

In [49]:
prp_summary = (
    prp_df.groupby(['Model_Slug', 'PRP_Name'], as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', std_affinity='std', n='count')
)
display(prp_summary.head(20))

,Model_Slug,PRP_Name,mean_affinity,std_affinity,n
0,gemini-3-flash-preview,CROF,35.34,25.095865,100
1,gemini-3-flash-preview,DER,34.48,30.271659,100
2,gemini-3-flash-preview,IBR,33.71,28.320780,100
3,gemini-3-flash-preview,Planning,50.75,27.327559,100
4,gemini-3-flash-preview,Stability,41.17,33.647326,100
5,gemini-3-flash-preview,System Services,35.56,22.290697,100
6,gemma4-31b-cloud,CROF,35.85,24.659858,100
7,gemma4-31b-cloud,DER,35.65,29.376131,100
8,gemma4-31b-cloud,IBR,40.88,26.909230,100
9,gemma4-31b-cloud,Planning,48.80,27.717186,100


In [50]:
# 
fig = px.scatter(
    prp_summary,
    color='PRP_Name',
    y='mean_affinity',
    x = 'Model_Slug',
    title='PRP Affinity Distribution by Model',
    # points = 'all',
 )
fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [51]:
prp_heat = prp_summary.pivot(index='Model_Slug', columns='PRP_Name', values='mean_affinity').sort_index()
fig = px.imshow(
    prp_heat,
    text_auto='.1f',
    color_continuous_scale='Viridis',
    title='Mean PRP Affinity (Model x PRP)',
    aspect='auto'
 )
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Model')
fig.show()

In [52]:
# Top-1 PRP agreement per abstract
prp_top1 = (
    prp_df.sort_values(['Model_Slug', 'Abstract_Index', 'LLM_Affinity'], ascending=[True, True, False])
    .groupby(['Model_Slug', 'Abstract_Index'], as_index=False)
    .first()[['Model_Slug', 'Abstract_Index', 'PRP_Name','LLM_Affinity']]
)

prp_agreement = (
    prp_top1.groupby('Abstract_Index')['PRP_Name']
    .nunique()
    .rename('n_unique_top1_prp')
    .reset_index()
)

print('Full agreement rate (all models same top-1 PRP):', (prp_agreement['n_unique_top1_prp'] == 1).mean())
display(prp_agreement['n_unique_top1_prp'].value_counts().sort_index().rename('n_abstracts'))

Full agreement rate (all models same top-1 PRP): 0.53


n_unique_top1_prp
1    53
2    33
3    13
4     1
Name: n_abstracts, dtype: int64

## RA Summaries

In [53]:
ra_df.head(20)

,Abstract_Index,Document Title,RA2025_ID,RA_Question,Cosine_x100,LLM_Affinity,LLM_Affinity_Reason,Model_Slug,Run_ID
0,1,Scenario Reduction Network Based on Wasserstei...,37,what studies and metrics are required to ident...,NaN,NaN,NaN,gemini-3-flash-preview,20260413_151017
1,1,Scenario Reduction Network Based on Wasserstei...,38,how should grid enhancing technologies be incl...,NaN,NaN,NaN,gemini-3-flash-preview,20260413_151017
2,1,Scenario Reduction Network Based on Wasserstei...,39,what technical models need to be added to long...,NaN,NaN,NaN,gemini-3-flash-preview,20260413_151017
3,1,Scenario Reduction Network Based on Wasserstei...,40,what studies and metrics are required to evalu...,NaN,NaN,NaN,gemini-3-flash-preview,20260413_151017
4,1,Scenario Reduction Network Based on Wasserstei...,41,what additional probabilistic planning methods...,NaN,NaN,NaN,gemini-3-flash-preview,20260413_151017
5,1,Scenario Reduction Network Based on Wasserstei...,42,what additional planning models and methods ar...,NaN,NaN,NaN,gemini-3-flash-preview,20260413_151017
6,1,Scenario Reduction Network Based on Wasserstei...,43,what changes can be incorporated into the tran...,NaN,NaN,NaN,gemini-3-flash-preview,20260413_151017
7,1,Scenario Reduction Network Based on Wasserstei...,44,"what methods, processes, and products (e.g. se...",NaN,NaN,NaN,gemini-3-flash-preview,20260413_151017
8,1,Scenario Reduction Network Based on Wasserstei...,45,what additional load and resource forecasting ...,NaN,NaN,NaN,gemini-3-flash-preview,20260413_151017
9,1,Scenario Reduction Network Based on Wasserstei...,46,"what additional methods, tools, and metrics ar...",NaN,NaN,NaN,gemini-3-flash-preview,20260413_151017


In [54]:
# fig = px.box(
#     ra_df,
#     x='Abstract_Index',
#     y='LLM_Affinity',
#     color='Model_Slug',
#     points='all',
#     # facet_row='PRP_Name',
#     # color = 'PRP_Name',
#     title=f'RA Affinity Distribution by Abstract'
#  )
# fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# # fig.update_xaxes(tickangle=45)
# fig.show()

In [55]:
fig = px.scatter(#prp_df[prp_df['Abstract_Index'].isin([1,8,20,19])]
    ra_df[ra_df['Abstract_Index'].isin([1,8,20,19])],
    # ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='Model_Slug',
    facet_col = 'Abstract_Index',
    # points='all',
    hover_data = 'Model_Slug',
    # facet_row='PRP_Name',
    # color = 'PRP_Name',
    # title=f'RA Affinity Distribution by Abstract'
 )
fig.update_layout(xaxis_title='RA question', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [56]:
ra_summary = (
    ra_df.groupby(['Model_Slug', 'RA2025_ID'], as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', std_affinity='std', n='count')
)
display(ra_summary.head(20))

,Model_Slug,RA2025_ID,mean_affinity,std_affinity,n
0,gemini-3-flash-preview,1,NaN,NaN,0
1,gemini-3-flash-preview,2,NaN,NaN,0
2,gemini-3-flash-preview,3,NaN,NaN,0
3,gemini-3-flash-preview,4,NaN,NaN,0
4,gemini-3-flash-preview,5,NaN,NaN,0
5,gemini-3-flash-preview,6,NaN,NaN,0
6,gemini-3-flash-preview,7,NaN,NaN,0
7,gemini-3-flash-preview,8,NaN,NaN,0
8,gemini-3-flash-preview,9,NaN,NaN,0
9,gemini-3-flash-preview,10,NaN,NaN,0


In [57]:
fig = px.box(
    prp_df,
    x='Model_Slug',
    y='LLM_Affinity',
    title='RA Affinity Distribution by Model'
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [58]:
fig = px.box(
    ra_df,
    x='Model_Slug',
    y='LLM_Affinity',
    title='RA Affinity Distribution by Model'
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [59]:
# Top-1 RA agreement per abstract
ra_top1 = (
    ra_df.sort_values(['Model_Slug', 'Abstract_Index', 'LLM_Affinity'], ascending=[True, True, False])
    .groupby(['Model_Slug', 'Abstract_Index'], as_index=False)
    .first()[['Model_Slug', 'Abstract_Index', 'RA2025_ID']]
)

ra_agreement = (
    ra_top1.groupby('Abstract_Index')['RA2025_ID']
    .nunique()
    .rename('n_unique_top1_ra')
    .reset_index()
)

print('Full agreement rate (all models same top-1 RA):', (ra_agreement['n_unique_top1_ra'] == 1).mean())
display(ra_agreement['n_unique_top1_ra'].value_counts().sort_index().rename('n_abstracts'))

Full agreement rate (all models same top-1 RA): 0.02


n_unique_top1_ra
1     2
2    16
3    41
4    34
5     7
Name: n_abstracts, dtype: int64

## Coherence Analysis: Model Agreement on Affinity Levels

Analyze whether all models assign similar affinity scores to the same questions/abstracts.

## Coherence Logic: Coefficient of Variation (CV)

The coefficient of variation measures how dispersed the model affinities are relative to their average value.  
For each abstract-target unit, we collect the affinity scores from all models and compute:

$$
\mu = \frac{1}{n}\sum_{i=1}^{n} x_i
$$

$$
\sigma = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\mu)^2}
$$

$$
CV = \frac{\sigma}{\mu + \varepsilon}
$$

where:

- $x_i$ is the affinity score assigned by model $i$
- $n$ is the number of models
- $\mu$ is the mean affinity
- $\sigma$ is the sample standard deviation
- $\varepsilon$ is a small constant added to avoid division by zero

Interpretation:

- Low CV means the models are in close agreement
- High CV means the models disagree more strongly
- CV can become unstable when the mean affinity is very small, so low-mean cases should be interpreted with caution

In [60]:
def affinity_level_fixed(score):
    if pd.isna(score):
        return pd.NA
    try:
        x = float(score)
    except Exception:
        return pd.NA
    if 0 <= x <= 40:
        return 'Low'
    if 40 < x <= 70:
        return 'Moderate'
    if 70 < x <= 100:
        return 'High'
    return pd.NA


def compute_unit_coherency(df, target_col, eps=1e-8, low_mean_threshold=5.0):
    required_cols = {'Abstract_Index', 'Model_Slug', target_col, 'LLM_Affinity'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    unit_stats = (
        df.groupby(['Abstract_Index', target_col], as_index=False)['LLM_Affinity']
        .agg(
            mean_affinity='mean',
            std_affinity='std',
            min_affinity='min',
            max_affinity='max',
            n_models='count',
            mad_affinity=lambda s: float(np.mean(np.abs(s - s.mean()))),
        )
    )
    unit_stats['cv'] = unit_stats['std_affinity'] / (unit_stats['mean_affinity'] + eps)
    unit_stats.loc[unit_stats['n_models'] < 2, 'cv'] = np.nan
    unit_stats['range'] = unit_stats['max_affinity'] - unit_stats['min_affinity']
    unit_stats['Affinity_Level'] = unit_stats['mean_affinity'].apply(affinity_level_fixed)
    unit_stats['cv_low_mean_flag'] = unit_stats['mean_affinity'] < float(low_mean_threshold)
    unit_stats['singleton_flag'] = unit_stats['n_models'] < 2
    return unit_stats


def summarize_coherency_by_level(unit_stats):
    level_order = pd.CategoricalDtype(categories=['Low', 'Moderate', 'High'], ordered=True)
    out = unit_stats.copy()
    out['Affinity_Level'] = out['Affinity_Level'].astype(level_order)

    level_summary = (
        out.dropna(subset=['Affinity_Level'])
        .groupby('Affinity_Level', as_index=False, observed=False)
        .agg(
            n_units=('Affinity_Level', 'count'),
            mean_cv=('cv', 'mean'),
            median_cv=('cv', 'median'),
            p90_cv=('cv', lambda s: float(np.percentile(s.dropna(), 90)) if s.notna().any() else np.nan),
            mean_mad=('mad_affinity', 'mean'),
            median_mad=('mad_affinity', 'median'),
            mean_range=('range', 'mean'),
            pct_low_mean_flag=('cv_low_mean_flag', 'mean'),
            pct_singleton=('singleton_flag', 'mean'),
        )
        .sort_values('Affinity_Level')
    )
    level_summary['pct_low_mean_flag'] = 100.0 * level_summary['pct_low_mean_flag']
    level_summary['pct_singleton'] = 100.0 * level_summary['pct_singleton']
    return level_summary


def compute_pairwise_disagreement_by_level(df, target_col):
    records = []
    for (abstract_idx, target_id), grp in df.groupby(['Abstract_Index', target_col]):
        grp2 = grp[['Model_Slug', 'LLM_Affinity']].dropna()
        if grp2['Model_Slug'].nunique() < 2:
            continue

        level = affinity_level_fixed(grp2['LLM_Affinity'].mean())
        rows = grp2.sort_values('Model_Slug')[['Model_Slug', 'LLM_Affinity']].to_records(index=False)

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                m1, a1 = rows[i]
                m2, a2 = rows[j]
                records.append(
                    {
                        'Abstract_Index': abstract_idx,
                        'Target_ID': target_id,
                        'Affinity_Level': level,
                        'Model_1': str(m1),
                        'Model_2': str(m2),
                        'Affinity_Difference': abs(float(a1) - float(a2)),
                    }
                )

    pairwise = pd.DataFrame(records)
    if pairwise.empty:
        return pairwise, pairwise

    pairwise_summary = (
        pairwise.dropna(subset=['Affinity_Level'])
        .groupby(['Affinity_Level', 'Model_1', 'Model_2'], as_index=False)
        .agg(mean_diff=('Affinity_Difference', 'mean'), median_diff=('Affinity_Difference', 'median'), std_diff=('Affinity_Difference', 'std'), n_pairs=('Affinity_Difference', 'count'))
        .sort_values(['Affinity_Level', 'mean_diff'], ascending=[True, False])
    )
    return pairwise, pairwise_summary

In [61]:
prp_coherency = compute_unit_coherency(prp_df, target_col='PRP_Name')

print("PRP Coherency Summary:")
print(f"Mean CV across all abstract-PRP pairs: {prp_coherency['cv'].mean():.3f}")
print(f"Abstracts with perfect agreement (CV=0): {(prp_coherency['cv'] == 0).sum()}")
print("\nBottom 10 coherent (high CV) abstract-PRP pairs:")
display(prp_coherency.nlargest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])
print("\nTop 10 coherent (low CV) abstract-PRP pairs:")
display(prp_coherency.nsmallest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])

PRP Coherency Summary:
Mean CV across all abstract-PRP pairs: 0.530
Abstracts with perfect agreement (CV=0): 0

Bottom 10 coherent (high CV) abstract-PRP pairs:


,Abstract_Index,PRP_Name,mean_affinity,cv,range
318,54,CROF,2.0,2.236068,10.0
13,3,DER,4.0,1.629801,15.0
199,34,DER,3.0,1.490712,10.0
202,34,Stability,3.0,1.490712,10.0
475,80,DER,3.0,1.490712,10.0
474,80,CROF,10.0,1.414214,30.0
205,35,DER,5.0,1.414214,15.0
479,80,System Services,7.0,1.392399,20.0
17,3,System Services,8.0,1.369306,20.0
336,57,CROF,6.0,1.369306,20.0



Top 10 coherent (low CV) abstract-PRP pairs:


,Abstract_Index,PRP_Name,mean_affinity,cv,range
262,44,Stability,90.4,0.009894,2.0
116,20,IBR,96.4,0.015732,3.0
123,21,Planning,94.6,0.016031,4.0
322,54,Stability,96.2,0.017081,3.0
454,76,Stability,95.8,0.018673,4.0
400,67,Stability,86.2,0.019062,3.0
483,81,Planning,91.2,0.019615,4.0
596,100,IBR,96.4,0.020222,4.0
598,100,Stability,87.2,0.024862,5.0
502,84,Stability,93.0,0.026339,6.0


In [62]:
prp_coherency

,Abstract_Index,PRP_Name,mean_affinity,std_affinity,min_affinity,max_affinity,n_models,mad_affinity,cv,range,Affinity_Level,cv_low_mean_flag,singleton_flag
0,1,CROF,24.0,14.317821,5.0,40.0,5,11.20,0.596576,35.0,Low,False,False
1,1,DER,19.0,6.519202,10.0,25.0,5,5.20,0.343116,15.0,Low,False,False
2,1,IBR,17.0,9.746794,5.0,30.0,5,7.60,0.573341,25.0,Low,False,False
3,1,Planning,79.6,11.760102,60.0,90.0,5,8.48,0.147740,30.0,High,False,False
4,1,Stability,18.0,7.582875,10.0,30.0,5,5.60,0.421271,20.0,Low,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,100,DER,22.6,6.985700,18.0,35.0,5,4.96,0.309102,17.0,Low,False,False
596,100,IBR,96.4,1.949359,95.0,99.0,5,1.68,0.020222,4.0,High,False,False
597,100,Planning,12.4,2.509980,10.0,15.0,5,2.08,0.202418,5.0,Low,False,False
598,100,Stability,87.2,2.167948,85.0,90.0,5,1.76,0.024862,5.0,High,False,False


In [63]:
fig1 = px.box(
    prp_df[prp_df['Abstract_Index'].isin(prp_coherency.nlargest(7, 'range')['Abstract_Index'])],
    y='LLM_Affinity',
    points='all',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution (top CV abstracts)'
 )
fig2 = px.box(
    prp_df[prp_df['Abstract_Index'].isin(prp_coherency.nsmallest(5, 'range')['Abstract_Index'])],
    y='LLM_Affinity',
    points='all',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution (low CV abstracts)'
 )
# fig.update_xaxes(range=[0.5,10.5])
fig1.update_layout(yaxis_title='LLM Affinity')
fig2.update_layout(yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig1.show()
fig2.show()

In [64]:
ra_coherency = compute_unit_coherency(ra_df, target_col='RA2025_ID')

print("RA Coherency Summary:")
print(f"Mean CV across all abstract-RA pairs: {ra_coherency['cv'].mean():.3f}")
print(f"Abstracts with perfect agreement (CV=0): {(ra_coherency['cv'] == 0).sum()}")
print("\nBottom 10 coherent (high CV) abstract-RA pairs:")
display(ra_coherency.nlargest(10, 'cv')[['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'range']])
print("\nTop 10 coherent (low CV) abstract-RA pairs:")
display(ra_coherency.nsmallest(10, 'cv')[['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'range']])

RA Coherency Summary:
Mean CV across all abstract-RA pairs: 1.208
Abstracts with perfect agreement (CV=0): 718

Bottom 10 coherent (high CV) abstract-RA pairs:


,Abstract_Index,RA2025_ID,mean_affinity,cv,range
387,9,4,22.50,2.0,90.0
1776,38,1,21.00,2.0,84.0
1950,41,31,20.00,2.0,80.0
1347,29,4,17.50,2.0,70.0
1731,37,4,17.50,2.0,70.0
3747,79,4,17.50,2.0,70.0
3939,83,4,17.50,2.0,70.0
4161,87,34,17.50,2.0,70.0
1059,23,4,16.25,2.0,65.0
1299,28,4,16.25,2.0,65.0



Top 10 coherent (low CV) abstract-RA pairs:


,Abstract_Index,RA2025_ID,mean_affinity,cv,range
240,6,1,0.0,0.0,0.0
241,6,2,0.0,0.0,0.0
242,6,3,0.0,0.0,0.0
243,6,4,0.0,0.0,0.0
244,6,5,0.0,0.0,0.0
245,6,6,0.0,0.0,0.0
246,6,7,0.0,0.0,0.0
247,6,8,0.0,0.0,0.0
249,6,10,0.0,0.0,0.0
250,6,11,0.0,0.0,0.0


In [65]:
# Abstract-level coherency: aggregate CV per abstract (for both PRP and RA)
prp_abstract_coherency = (
    prp_coherency.groupby('Abstract_Index', as_index=False)['cv']
    .agg(mean_cv='mean', max_cv='max', min_cv='min', std_cv='std', n_pairs='count')
    .sort_values('mean_cv', ascending=True)
)

ra_abstract_coherency = (
    ra_coherency.groupby('Abstract_Index', as_index=False)['cv']
    .agg(mean_cv='mean', max_cv='max', min_cv='min', std_cv='std', n_pairs='count')
    .sort_values('mean_cv', ascending=True)
)

print("PRP Abstract-level Coherency (Average CV per Abstract):")
print(f"Most coherent abstracts (lowest mean CV):")
display(prp_abstract_coherency.head(10))
print(f"\nLeast coherent abstracts (highest mean CV):")
display(prp_abstract_coherency.tail(10))

print("\n" + "="*60)
print("\nRA Abstract-level Coherency (Average CV per Abstract):")
print(f"Most coherent abstracts (lowest mean CV):")
display(ra_abstract_coherency.head(10))
print(f"\nLeast coherent abstracts (highest mean CV):")
display(ra_abstract_coherency.tail(10))

PRP Abstract-level Coherency (Average CV per Abstract):
Most coherent abstracts (lowest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
18,19,0.224516,0.464498,0.067448,0.186453,6
19,20,0.239587,0.638877,0.015732,0.243467,6
55,56,0.263999,0.697217,0.094596,0.226566,6
99,100,0.278564,1.000000,0.020222,0.370170,6
21,22,0.286214,0.495137,0.046010,0.181496,6
86,87,0.294670,0.548319,0.041390,0.193450,6
80,81,0.301228,0.502930,0.019615,0.198540,6
47,48,0.303474,0.595067,0.098432,0.191299,6
72,73,0.322973,0.813716,0.033981,0.265135,6
64,65,0.326457,0.502727,0.105010,0.178653,6



Least coherent abstracts (highest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
95,96,0.728502,1.068632,0.166820,0.356421,6
92,93,0.734362,1.125771,0.370847,0.266691,6
36,37,0.742371,0.885344,0.617208,0.106805,6
60,61,0.747969,1.195229,0.029768,0.413096,6
73,74,0.755270,1.240884,0.311683,0.343977,6
46,47,0.760557,0.958315,0.103235,0.338079,6
2,3,0.981099,1.629801,0.032789,0.549543,6
53,54,1.010918,2.236068,0.017081,0.730715,6
79,80,1.088447,1.490712,0.094808,0.521149,6
33,34,1.197856,1.490712,0.644887,0.325787,6




RA Abstract-level Coherency (Average CV per Abstract):
Most coherent abstracts (lowest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
53,54,0.282166,1.732051,0.0,0.506941,39
89,90,0.292816,2.000000,0.0,0.637202,48
77,78,0.310872,2.000000,0.0,0.605255,48
73,74,0.311907,2.000000,0.0,0.592852,48
31,32,0.356326,1.732051,0.0,0.604435,41
19,20,0.357626,1.732051,0.0,0.620360,48
56,57,0.372671,2.000000,0.0,0.537846,48
33,34,0.387714,2.000000,0.0,0.736680,48
99,100,0.430937,1.732051,0.0,0.613269,39
70,71,0.456161,1.732051,0.0,0.660396,48



Least coherent abstracts (highest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
30,31,1.708901,2.0,0.434923,0.436275,48
24,25,1.710259,2.0,0.000000,0.523655,48
42,43,1.736311,2.0,0.326829,0.394052,48
82,83,1.745958,2.0,0.328497,0.400606,48
76,77,1.760723,2.0,0.328497,0.474703,48
48,49,1.783583,2.0,0.716512,0.371622,48
68,69,1.845410,2.0,0.783426,0.316682,48
40,41,1.851552,2.0,0.737820,0.326395,48
29,30,1.865843,2.0,0.764213,0.329189,48
67,68,1.966376,2.0,1.200000,0.143091,48


In [66]:
# Visualization: Coherency Distribution
fig = px.histogram(
    prp_coherency[prp_coherency['cv'].notna()],
    x='cv',
    # nbins=30,
    # facet_col = 'PRP_Name',
    title='PRP Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=prp_coherency['cv'].mean(), line_dash="dash", line_color="red", 
              annotation_text=f"Mean CV: {prp_coherency['cv'].mean():.3f}")

fig.update_xaxes(range=[0,2])
fig.show()

fig = px.histogram(
    ra_coherency[ra_coherency['cv'].notna()],
    x='cv',
    nbins=30,
    title='RA Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=ra_coherency['cv'].mean(), line_dash="dash", line_color="red",
              annotation_text=f"Mean CV: {ra_coherency['cv'].mean():.3f}")
fig.show()

In [67]:
# Abstract-level coherency scatter plots
fig = px.box(
    prp_coherency,
    x='Abstract_Index',
    y='cv',
    title='PRP Abstract-level Coherency (Lower CV = Higher Agreement)',
    points ='all',
)
fig.update_traces(boxmean=True)  # set to 'sd' to show mean + std
fig.show()

fig = px.box(
    ra_coherency,
    x='Abstract_Index',
    y='cv',
    title='RA Abstract-level Coherency (Lower CV = Higher Agreement)',
)
fig.update_traces(boxmean=True)  # set to 'sd' to show mean + std
fig.show()


In [68]:
# Abstract-level coherency scatter plots
fig = px.scatter(
    prp_abstract_coherency,
    x='Abstract_Index',
    y='mean_cv',
    size='n_pairs',
    color='mean_cv',
    hover_data=['max_cv', 'std_cv'],
    title='PRP Abstract-level Coherency (Lower CV = Higher Agreement)',
    labels={'mean_cv': 'Mean CV (Coherency)', 'Abstract_Index': 'Abstract Index'},
    color_continuous_scale='RdYlGn_r'
)
fig.show()

fig = px.scatter(
    ra_abstract_coherency,
    x='Abstract_Index',
    y='mean_cv',
    size='n_pairs',
    color='mean_cv',
    hover_data=['max_cv', 'std_cv'],
    title='RA Abstract-level Coherency (Lower CV = Higher Agreement)',
    labels={'mean_cv': 'Mean CV (Coherency)', 'Abstract_Index': 'Abstract Index'},
    color_continuous_scale='RdYlGn_r'
)
fig.show()

In [69]:
# Pairwise Model Coherency: Correlation between model pairs per abstract-PRP
# This shows which model pairs agree most on affinity levels

from itertools import combinations

# For PRP: compute correlation between each pair of models per abstract
prp_model_pairs = []
for abstract_idx in prp_df['Abstract_Index'].unique():
    for prp_name in prp_df[prp_df['Abstract_Index'] == abstract_idx]['PRP_Name'].unique():
        subset = prp_df[(prp_df['Abstract_Index'] == abstract_idx) & (prp_df['PRP_Name'] == prp_name)]
        
        if len(subset) >= 2:
            models = subset['Model_Slug'].unique()
            for model1, model2 in combinations(models, 2):
                val1 = subset[subset['Model_Slug'] == model1]['LLM_Affinity'].values
                val2 = subset[subset['Model_Slug'] == model2]['LLM_Affinity'].values
                
                if len(val1) > 0 and len(val2) > 0:
                    diff = abs(val1[0] - val2[0])
                    prp_model_pairs.append({
                        'Abstract_Index': abstract_idx,
                        'PRP_Name': prp_name,
                        'Model_1': min(model1, model2),
                        'Model_2': max(model1, model2),
                        'Affinity_Difference': diff
                    })

prp_pairwise = pd.DataFrame(prp_model_pairs)

if len(prp_pairwise) > 0:
    print("PRP Model Pair Agreement Analysis:")
    print("\nModel pairs with highest average disagreement (highest mean affinity difference):")
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', std_diff='std', n_pairs='count'
    ).sort_values('mean_diff', ascending=False)
    display(pair_stats.head(10))
    
    print("\nModel pairs with lowest average disagreement (highest mean agreement):")
    display(pair_stats.sort_values('mean_diff', ascending=True).head(10))

PRP Model Pair Agreement Analysis:

Model pairs with highest average disagreement (highest mean affinity difference):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
9,gpt-5.4-nano,ministral-3-14b-cloud,20.531667,16.894616,600
5,gemma4-31b-cloud,gpt-5.4-nano,18.080000,15.111724,600
2,gemini-3-flash-preview,gpt-5.4-nano,16.835000,15.113299,600
8,gpt-5.4-mini,ministral-3-14b-cloud,16.490000,14.504155,600
3,gemini-3-flash-preview,ministral-3-14b-cloud,16.040000,14.897033,600
6,gemma4-31b-cloud,ministral-3-14b-cloud,14.118333,13.433606,600
4,gemma4-31b-cloud,gpt-5.4-mini,13.721667,12.305415,600
1,gemini-3-flash-preview,gpt-5.4-mini,12.863333,12.496597,600
7,gpt-5.4-mini,gpt-5.4-nano,11.978333,12.904084,600
0,gemini-3-flash-preview,gemma4-31b-cloud,8.051667,7.208717,600



Model pairs with lowest average disagreement (highest mean agreement):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
0,gemini-3-flash-preview,gemma4-31b-cloud,8.051667,7.208717,600
7,gpt-5.4-mini,gpt-5.4-nano,11.978333,12.904084,600
1,gemini-3-flash-preview,gpt-5.4-mini,12.863333,12.496597,600
4,gemma4-31b-cloud,gpt-5.4-mini,13.721667,12.305415,600
6,gemma4-31b-cloud,ministral-3-14b-cloud,14.118333,13.433606,600
3,gemini-3-flash-preview,ministral-3-14b-cloud,16.040000,14.897033,600
8,gpt-5.4-mini,ministral-3-14b-cloud,16.490000,14.504155,600
2,gemini-3-flash-preview,gpt-5.4-nano,16.835000,15.113299,600
5,gemma4-31b-cloud,gpt-5.4-nano,18.080000,15.111724,600
9,gpt-5.4-nano,ministral-3-14b-cloud,20.531667,16.894616,600


In [70]:
# For RA: compute correlation between each pair of models per abstract
ra_model_pairs = []
for abstract_idx in ra_df['Abstract_Index'].unique():
    for ra_id in ra_df[ra_df['Abstract_Index'] == abstract_idx]['RA2025_ID'].unique():
        subset = ra_df[(ra_df['Abstract_Index'] == abstract_idx) & (ra_df['RA2025_ID'] == ra_id)]
        
        if len(subset) >= 2:
            models = subset['Model_Slug'].unique()
            for model1, model2 in combinations(models, 2):
                val1 = subset[subset['Model_Slug'] == model1]['LLM_Affinity'].values
                val2 = subset[subset['Model_Slug'] == model2]['LLM_Affinity'].values
                
                if len(val1) > 0 and len(val2) > 0:
                    diff = abs(val1[0] - val2[0])
                    ra_model_pairs.append({
                        'Abstract_Index': abstract_idx,
                        'RA2025_ID': ra_id,
                        'Model_1': min(model1, model2),
                        'Model_2': max(model1, model2),
                        'Affinity_Difference': diff
                    })

ra_pairwise = pd.DataFrame(ra_model_pairs)

if len(ra_pairwise) > 0:
    print("RA Model Pair Agreement Analysis:")
    print("\nModel pairs with highest average disagreement (highest mean affinity difference):")
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', std_diff='std', n_pairs='count'
    ).sort_values('mean_diff', ascending=False)
    display(pair_stats_ra.head(10))
    
    print("\nModel pairs with lowest average disagreement (highest mean agreement):")
    display(pair_stats_ra.sort_values('mean_diff', ascending=True).head(10))

RA Model Pair Agreement Analysis:

Model pairs with highest average disagreement (highest mean affinity difference):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
9,gpt-5.4-nano,ministral-3-14b-cloud,28.689069,19.506894,3586
8,gpt-5.4-mini,ministral-3-14b-cloud,23.615231,17.365617,4018
6,gemma4-31b-cloud,ministral-3-14b-cloud,21.056802,16.281657,4102
5,gemma4-31b-cloud,gpt-5.4-nano,13.998256,20.194476,4014
4,gemma4-31b-cloud,gpt-5.4-mini,11.060404,16.109118,4503
7,gpt-5.4-mini,gpt-5.4-nano,9.271992,17.213417,3956
0,gemini-3-flash-preview,gemma4-31b-cloud,NaN,NaN,0
1,gemini-3-flash-preview,gpt-5.4-mini,NaN,NaN,0
2,gemini-3-flash-preview,gpt-5.4-nano,NaN,NaN,0
3,gemini-3-flash-preview,ministral-3-14b-cloud,NaN,NaN,0



Model pairs with lowest average disagreement (highest mean agreement):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
7,gpt-5.4-mini,gpt-5.4-nano,9.271992,17.213417,3956
4,gemma4-31b-cloud,gpt-5.4-mini,11.060404,16.109118,4503
5,gemma4-31b-cloud,gpt-5.4-nano,13.998256,20.194476,4014
6,gemma4-31b-cloud,ministral-3-14b-cloud,21.056802,16.281657,4102
8,gpt-5.4-mini,ministral-3-14b-cloud,23.615231,17.365617,4018
9,gpt-5.4-nano,ministral-3-14b-cloud,28.689069,19.506894,3586
0,gemini-3-flash-preview,gemma4-31b-cloud,NaN,NaN,0
1,gemini-3-flash-preview,gpt-5.4-mini,NaN,NaN,0
2,gemini-3-flash-preview,gpt-5.4-nano,NaN,NaN,0
3,gemini-3-flash-preview,ministral-3-14b-cloud,NaN,NaN,0


In [71]:
# Model Agreement Heatmaps: Show which models have smallest disagreements
if len(prp_pairwise) > 0:
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', n_pairs='count'
    ).sort_values('mean_diff', ascending=True)
    
    # Create a symmetrical matrix for heatmap
    all_models = sorted(set(list(pair_stats['Model_1'].unique()) + list(pair_stats['Model_2'].unique())))
    model_agreement_matrix = pd.DataFrame(0.0, index=all_models, columns=all_models)
    
    for _, row in pair_stats.iterrows():
        model_agreement_matrix.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
        model_agreement_matrix.loc[row['Model_2'], row['Model_1']] = row['mean_diff']
    
    fig = px.imshow(
        model_agreement_matrix,
        text_auto='.2f',
        color_continuous_scale='RdYlGn_r',
        title='PRP Model Pairwise Disagreement (Lower = Better Agreement)',
        labels={'color': 'Mean Affinity Difference'},
        aspect='auto'
    )
    fig.update_layout(xaxis_title='Model', yaxis_title='Model')
    fig.show()

if len(ra_pairwise) > 0:
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', n_pairs='count'
    ).sort_values('mean_diff', ascending=True)
    
    all_models_ra = sorted(set(list(pair_stats_ra['Model_1'].unique()) + list(pair_stats_ra['Model_2'].unique())))
    model_agreement_matrix_ra = pd.DataFrame(0.0, index=all_models_ra, columns=all_models_ra)
    
    for _, row in pair_stats_ra.iterrows():
        model_agreement_matrix_ra.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
        model_agreement_matrix_ra.loc[row['Model_2'], row['Model_1']] = row['mean_diff']
    
    fig = px.imshow(
        model_agreement_matrix_ra,
        text_auto='.2f',
        color_continuous_scale='RdYlGn_r',
        title='RA Model Pairwise Disagreement (Lower = Better Agreement)',
        labels={'color': 'Mean Affinity Difference'},
        aspect='auto'
    )
    fig.update_layout(xaxis_title='Model', yaxis_title='Model')
    fig.show()

## Coherency Key Insights

Summary of model agreement levels on affinity assignments.

In [72]:
# Summary statistics on model coherency
print("=" * 70)
print("COHERENCY ANALYSIS SUMMARY")
print("=" * 70)

print("\n📊 PRP COHERENCY:")
print(f"  • Mean CV (lower = more agreement): {prp_coherency['cv'].mean():.3f}")
print(f"  • Median CV: {prp_coherency['cv'].median():.3f}")
print(f"  • Perfect agreement cases (CV=0): {(prp_coherency['cv'] == 0).sum()} / {len(prp_coherency)}")
print(f"  • High disagreement cases (CV>1): {(prp_coherency['cv'] > 1).sum()} / {len(prp_coherency)}")
print(f"  • Range of affinities (mean): {prp_coherency['range'].mean():.3f}")
print(f"  • Most coherent abstract: {prp_abstract_coherency.iloc[0]['Abstract_Index']} (mean CV: {prp_abstract_coherency.iloc[0]['mean_cv']:.3f})")
print(f"  • Least coherent abstract: {prp_abstract_coherency.iloc[-1]['Abstract_Index']} (mean CV: {prp_abstract_coherency.iloc[-1]['mean_cv']:.3f})")

print("\n📊 RA COHERENCY:")
print(f"  • Mean CV (lower = more agreement): {ra_coherency['cv'].mean():.3f}")
print(f"  • Median CV: {ra_coherency['cv'].median():.3f}")
print(f"  • Perfect agreement cases (CV=0): {(ra_coherency['cv'] == 0).sum()} / {len(ra_coherency)}")
print(f"  • High disagreement cases (CV>1): {(ra_coherency['cv'] > 1).sum()} / {len(ra_coherency)}")
print(f"  • Range of affinities (mean): {ra_coherency['range'].mean():.3f}")
print(f"  • Most coherent abstract: {ra_abstract_coherency.iloc[0]['Abstract_Index']} (mean CV: {ra_abstract_coherency.iloc[0]['mean_cv']:.3f})")
print(f"  • Least coherent abstract: {ra_abstract_coherency.iloc[-1]['Abstract_Index']} (mean CV: {ra_abstract_coherency.iloc[-1]['mean_cv']:.3f})")

if len(prp_pairwise) > 0:
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].mean()
    print("\n🔗 PRP MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):")
    for i, (_, row) in enumerate(pair_stats.sort_values('Affinity_Difference').head(3).iterrows(), 1):
        print(f"  {i}. {row['Model_1']} ↔ {row['Model_2']}: {row['Affinity_Difference']:.3f}")

if len(ra_pairwise) > 0:
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].mean()
    print("\n🔗 RA MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):")
    for i, (_, row) in enumerate(pair_stats_ra.sort_values('Affinity_Difference').head(3).iterrows(), 1):
        print(f"  {i}. {row['Model_1']} ↔ {row['Model_2']}: {row['Affinity_Difference']:.3f}")

print("\n" + "=" * 70)

COHERENCY ANALYSIS SUMMARY

📊 PRP COHERENCY:
  • Mean CV (lower = more agreement): 0.530
  • Median CV: 0.525
  • Perfect agreement cases (CV=0): 0 / 600
  • High disagreement cases (CV>1): 38 / 600
  • Range of affinities (mean): 30.643
  • Most coherent abstract: 19.0 (mean CV: 0.225)
  • Least coherent abstract: 34.0 (mean CV: 1.198)

📊 RA COHERENCY:
  • Mean CV (lower = more agreement): 1.208
  • Median CV: 1.277
  • Perfect agreement cases (CV=0): 718 / 4800
  • High disagreement cases (CV>1): 2951 / 4800
  • Range of affinities (mean): 29.373
  • Most coherent abstract: 54.0 (mean CV: 0.282)
  • Least coherent abstract: 68.0 (mean CV: 1.966)

🔗 PRP MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):
  1. gemini-3-flash-preview ↔ gemma4-31b-cloud: 8.052
  2. gpt-5.4-mini ↔ gpt-5.4-nano: 11.978
  3. gemini-3-flash-preview ↔ gpt-5.4-mini: 12.863

🔗 RA MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):
  1. gpt-5.4-mini ↔ gpt-5.4-nano: 9.272
  2. gemma4-31b-cloud ↔ gpt-5.4-min

In [73]:
# Build Affinity_level at the grouped granularity: (Abstract_Index, PRP_Name, Model_Slug)
prp_group_level = (
    prp_df.groupby(['Abstract_Index', 'PRP_Name', 'Model_Slug'], as_index=False)['LLM_Affinity']
    .mean()
    .rename(columns={'LLM_Affinity': 'group_mean_affinity'})
)
prp_group_level['Affinity_level'] = prp_group_level['group_mean_affinity'].apply(affinity_level_fixed)

# Merge the grouped label back to all rows
prp_df = prp_df.drop(columns=['Affinity_level'], errors='ignore').merge(
    prp_group_level[['Abstract_Index', 'PRP_Name', 'Model_Slug', 'Affinity_level']],
    on=['Abstract_Index', 'PRP_Name', 'Model_Slug'],
    how='left',
)

display(prp_df[['Abstract_Index', 'PRP_Name', 'Model_Slug', 'LLM_Affinity', 'Affinity_level']].head(10))

,Abstract_Index,PRP_Name,Model_Slug,LLM_Affinity,Affinity_level
0,1,IBR,gemini-3-flash-preview,20.0,Low
1,1,Stability,gemini-3-flash-preview,15.0,Low
2,1,DER,gemini-3-flash-preview,25.0,Low
3,1,CROF,gemini-3-flash-preview,35.0,Low
4,1,System Services,gemini-3-flash-preview,45.0,Moderate
5,1,Planning,gemini-3-flash-preview,85.0,High
6,2,IBR,gemini-3-flash-preview,45.0,Moderate
7,2,Stability,gemini-3-flash-preview,85.0,High
8,2,DER,gemini-3-flash-preview,20.0,Low
9,2,CROF,gemini-3-flash-preview,10.0,Low


In [74]:
# Agreement of Affinity_level across models for each (Abstract_Index, PRP_Name)
agreement_counts = (
    prp_group_level.groupby(['Abstract_Index', 'PRP_Name', 'Affinity_level'])['Model_Slug']
    .nunique()
    .rename('n_models_level')
    .reset_index()
)

agreement_wide = (
    agreement_counts.pivot_table(
        index=['Abstract_Index', 'PRP_Name'],
        columns='Affinity_level',
        values='n_models_level',
        fill_value=0,
    )
    .reset_index()
)

# Ensure consistent columns exist
for col in ['Low', 'Moderate', 'High']:
    if col not in agreement_wide.columns:
        agreement_wide[col] = 0

# Total number of models available per (Abstract_Index, PRP_Name)
n_models_total = (
    prp_group_level.groupby(['Abstract_Index', 'PRP_Name'])['Model_Slug']
    .nunique()
    .rename('n_models_total')
    .reset_index()
)
agreement_wide = agreement_wide.merge(
    n_models_total,
    on=['Abstract_Index', 'PRP_Name'],
    how='left',
)

agreement_wide['max_models_same_level'] = agreement_wide[['Low', 'Moderate', 'High']].max(axis=1)
agreement_wide['agreement_ratio'] = agreement_wide['max_models_same_level'] / agreement_wide['n_models_total']
agreement_wide['full_agreement'] = agreement_wide['max_models_same_level'] == agreement_wide['n_models_total']

# Table: how many models assigned each level for every abstract-PRP pair
display(agreement_wide.sort_values(['agreement_ratio', 'max_models_same_level'], ascending=[False, False]).head(30))

# Example lookup for one abstract + PRP
abstract_id = 20
prp_name = 'Planning'  # change to the PRP of interest
display(
    agreement_wide[
        (agreement_wide['Abstract_Index'] == abstract_id) &
        (agreement_wide['PRP_Name'] == prp_name)
    ]
)

,Abstract_Index,PRP_Name,High,Low,Moderate,n_models_total,max_models_same_level,agreement_ratio,full_agreement
0,1,CROF,0.0,5.0,0.0,5,5.0,1.0,True
1,1,DER,0.0,5.0,0.0,5,5.0,1.0,True
2,1,IBR,0.0,5.0,0.0,5,5.0,1.0,True
4,1,Stability,0.0,5.0,0.0,5,5.0,1.0,True
6,2,CROF,0.0,5.0,0.0,5,5.0,1.0,True
7,2,DER,0.0,5.0,0.0,5,5.0,1.0,True
9,2,Planning,0.0,5.0,0.0,5,5.0,1.0,True
10,2,Stability,5.0,0.0,0.0,5,5.0,1.0,True
11,2,System Services,0.0,5.0,0.0,5,5.0,1.0,True
12,3,CROF,0.0,5.0,0.0,5,5.0,1.0,True


,Abstract_Index,PRP_Name,High,Low,Moderate,n_models_total,max_models_same_level,agreement_ratio,full_agreement
117,20,Planning,0.0,5.0,0.0,5,5.0,1.0,True


In [75]:
# Heatmap: agreement ratio by Abstract and PRP
agreement_ratio_heat = (
    agreement_wide.pivot(columns='Abstract_Index', index='PRP_Name', values='agreement_ratio')
    .sort_index()
)

fig = px.imshow(
    agreement_ratio_heat,
    text_auto='.2f',
    zmin=0,
    zmax=1,
    color_continuous_scale='RdYlGn',
    title='Affinity Level Agreement Ratio Across Models (PRP x Abstract)',
    labels={'color': 'Agreement ratio'},
    aspect='auto',
)
fig.update_layout(yaxis_title='PRP Name', xaxis_title='Abstract Index')
fig.show()

In [76]:
(agreement_wide['agreement_ratio'] == 1).sum()

np.int64(341)

## Coherency by Fixed Affinity Levels (CV + MAD)

This section extends coherency analysis with fixed affinity bands and a robust companion metric:

- Low: 0-40
- Moderate: 41-70
- High: 71-100

Metrics per abstract-target unit across models:
- CV = std / mean (with epsilon)
- MAD = mean absolute deviation from cross-model mean

CV can be unstable when mean affinity is very low, so a low-mean caution flag is included.

In [77]:
# PRP coherency by fixed levels
prp_unit_coherency_lvl = compute_unit_coherency(prp_df, target_col='PRP_Name')
prp_level_summary = summarize_coherency_by_level(prp_unit_coherency_lvl)

print('PRP level-wise coherency summary (CV + MAD):')
display(prp_level_summary)

print('\nMost incoherent PRP units by CV:')
display(
    prp_unit_coherency_lvl.sort_values('cv', ascending=False)[
        ['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

print('\nMost incoherent PRP units by MAD:')
display(
    prp_unit_coherency_lvl.sort_values('mad_affinity', ascending=False)[
        ['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

fig = px.box(
    prp_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='cv',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='PRP Coherency by Affinity Level (CV)',
    points='all',
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='CV across models')
fig.show()

fig = px.box(
    prp_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='mad_affinity',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='PRP Coherency by Affinity Level (MAD)',
    points='all',
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='MAD across models')
fig.show()

PRP level-wise coherency summary (CV + MAD):


,Affinity_Level,n_units,mean_cv,median_cv,p90_cv,mean_mad,median_mad,mean_range,pct_low_mean_flag,pct_singleton
0,Low,405,0.667570,0.643585,1.000000,9.025975,7.92,28.711111,1.975309,0.0
1,Moderate,97,0.378079,0.384678,0.567208,15.538969,15.76,48.484536,0.000000,0.0
2,High,98,0.109612,0.096208,0.228509,6.417959,5.56,20.969388,0.000000,0.0



Most incoherent PRP units by CV:


,Abstract_Index,PRP_Name,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
318,54,CROF,2.0,2.236068,3.20,10.0,5,True
13,3,DER,4.0,1.629801,4.80,15.0,5,True
202,34,Stability,3.0,1.490712,3.60,10.0,5,True
199,34,DER,3.0,1.490712,3.60,10.0,5,True
475,80,DER,3.0,1.490712,3.60,10.0,5,True
474,80,CROF,10.0,1.414214,12.00,30.0,5,False
205,35,DER,5.0,1.414214,6.00,15.0,5,False
479,80,System Services,7.0,1.392399,8.40,20.0,5,False
17,3,System Services,8.0,1.369306,9.60,20.0,5,False
336,57,CROF,6.0,1.369306,5.60,20.0,5,False



Most incoherent PRP units by MAD:


,Abstract_Index,PRP_Name,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
192,33,CROF,41.0,0.815342,29.20,65.0,5,False
320,54,IBR,63.2,0.600787,28.56,88.0,5,False
234,40,CROF,52.6,0.645072,28.08,70.0,5,False
162,28,CROF,35.4,0.899220,27.68,65.0,5,False
216,37,CROF,37.2,0.885344,27.44,75.0,5,False
252,43,CROF,37.4,0.782423,24.88,65.0,5,False
432,73,CROF,36.4,0.813716,24.88,70.0,5,False
137,23,System Services,51.8,0.602552,24.64,70.0,5,False
386,65,IBR,63.6,0.502727,23.68,75.0,5,False
167,28,System Services,38.0,0.797980,23.60,77.0,5,False


In [78]:
# RA coherency by fixed levels
ra_unit_coherency_lvl = compute_unit_coherency(ra_df, target_col='RA2025_ID')
ra_level_summary = summarize_coherency_by_level(ra_unit_coherency_lvl)

print('RA level-wise coherency summary (CV + MAD):')
display(ra_level_summary)

print('\nMost incoherent RA units by CV:')
display(
    ra_unit_coherency_lvl.sort_values('cv', ascending=False)[
        ['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

print('\nMost incoherent RA units by MAD:')
display(
    ra_unit_coherency_lvl.sort_values('mad_affinity', ascending=False)[
        ['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

fig = px.box(
    ra_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='cv',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='RA Coherency by Affinity Level (CV)'
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='CV across models')
fig.show()

fig = px.box(
    ra_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='mad_affinity',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='RA Coherency by Affinity Level (MAD)'
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='MAD across models')
fig.show()

RA level-wise coherency summary (CV + MAD):


,Affinity_Level,n_units,mean_cv,median_cv,p90_cv,mean_mad,median_mad,mean_range,pct_low_mean_flag,pct_singleton
0,Low,4338,1.286557,1.414214,2.000000,10.224207,9.375,27.041724,30.889811,1.221761
1,Moderate,372,0.522059,0.511135,0.762083,19.891988,20.250,56.661290,0.000000,1.075269
2,High,81,0.178642,0.187742,0.340080,10.213992,10.750,28.876543,0.000000,0.000000



Most incoherent RA units by CV:


,Abstract_Index,RA2025_ID,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
387,9,4,22.50,2.0,33.750,90.0,4,False
1776,38,1,21.00,2.0,31.500,84.0,4,False
1950,41,31,20.00,2.0,30.000,80.0,4,False
1731,37,4,17.50,2.0,26.250,70.0,4,False
3747,79,4,17.50,2.0,26.250,70.0,4,False
4161,87,34,17.50,2.0,26.250,70.0,4,False
3939,83,4,17.50,2.0,26.250,70.0,4,False
1347,29,4,17.50,2.0,26.250,70.0,4,False
3960,83,25,16.25,2.0,24.375,65.0,4,False
4371,92,4,16.25,2.0,24.375,65.0,4,False



Most incoherent RA units by MAD:


,Abstract_Index,RA2025_ID,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
3570,75,19,30.000000,1.732051,40.000000,90.0,3,False
3891,82,4,38.750000,1.155901,38.750000,80.0,4,False
4641,97,34,38.250000,1.168869,38.250000,85.0,4,False
3589,75,38,43.250000,1.027689,38.250000,85.0,4,False
1971,42,4,46.250000,0.914670,36.250000,80.0,4,False
1222,26,23,36.250000,1.156073,36.250000,75.0,4,False
2931,62,4,39.000000,1.072845,36.000000,80.0,4,False
4068,85,37,40.750000,1.018405,35.750000,78.0,4,False
1985,42,18,35.000000,1.160577,35.000000,75.0,4,False
3921,82,34,35.000000,1.154701,35.000000,70.0,4,False


In [79]:
# Pairwise model disagreement by level (PRP)
prp_pairwise_lvl, prp_pairwise_lvl_summary = compute_pairwise_disagreement_by_level(prp_df, target_col='PRP_Name')

if not prp_pairwise_lvl_summary.empty:
    print('PRP pairwise disagreement by level (highest mean difference):')
    display(prp_pairwise_lvl_summary.head(20))

    for level in ['Low', 'Moderate', 'High']:
        level_df = prp_pairwise_lvl_summary[prp_pairwise_lvl_summary['Affinity_Level'] == level]
        if level_df.empty:
            continue

        models = sorted(set(level_df['Model_1']).union(level_df['Model_2']))
        mat = pd.DataFrame(np.nan, index=models, columns=models)
        np.fill_diagonal(mat.values, 0.0)

        for _, row in level_df.iterrows():
            mat.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
            mat.loc[row['Model_2'], row['Model_1']] = row['mean_diff']

        fig = px.imshow(
            mat,
            text_auto='.2f',
            color_continuous_scale='RdYlGn_r',
            title=f'PRP Pairwise Disagreement ({level})',
            labels={'color': 'Mean absolute difference'},
            aspect='auto'
        )
        fig.update_layout(xaxis_title='Model', yaxis_title='Model')
        fig.show()
else:
    print('No PRP pairwise level summary available.')

PRP pairwise disagreement by level (highest mean difference):


,Affinity_Level,Model_1,Model_2,mean_diff,median_diff,std_diff,n_pairs
2,High,gemini-3-flash-preview,gpt-5.4-nano,15.551020,11.0,14.813704,98
5,High,gemma4-31b-cloud,gpt-5.4-nano,15.265306,10.0,15.546771,98
9,High,gpt-5.4-nano,ministral-3-14b-cloud,14.867347,10.0,14.409073,98
7,High,gpt-5.4-mini,gpt-5.4-nano,14.795918,11.5,13.927618,98
3,High,gemini-3-flash-preview,ministral-3-14b-cloud,7.173469,5.0,7.602259,98
1,High,gemini-3-flash-preview,gpt-5.4-mini,7.000000,5.0,7.251022,98
8,High,gpt-5.4-mini,ministral-3-14b-cloud,6.540816,6.0,4.970362,98
6,High,gemma4-31b-cloud,ministral-3-14b-cloud,5.928571,5.0,5.919129,98
4,High,gemma4-31b-cloud,gpt-5.4-mini,5.244898,3.5,4.076741,98
0,High,gemini-3-flash-preview,gemma4-31b-cloud,4.857143,3.0,5.581634,98


In [80]:
# Pairwise model disagreement by level (RA)
ra_pairwise_lvl, ra_pairwise_lvl_summary = compute_pairwise_disagreement_by_level(ra_df, target_col='RA2025_ID')

if not ra_pairwise_lvl_summary.empty:
    print('RA pairwise disagreement by level (highest mean difference):')
    display(ra_pairwise_lvl_summary.head(20))

    for level in ['Low', 'Moderate', 'High']:
        level_df = ra_pairwise_lvl_summary[ra_pairwise_lvl_summary['Affinity_Level'] == level]
        if level_df.empty:
            continue

        models = sorted(set(level_df['Model_1']).union(level_df['Model_2']))
        mat = pd.DataFrame(np.nan, index=models, columns=models)
        np.fill_diagonal(mat.values, 0.0)

        for _, row in level_df.iterrows():
            mat.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
            mat.loc[row['Model_2'], row['Model_1']] = row['mean_diff']

        fig = px.imshow(
            mat,
            text_auto='.2f',
            color_continuous_scale='RdYlGn_r',
            title=f'RA Pairwise Disagreement ({level})',
            labels={'color': 'Mean absolute difference'},
            aspect='auto'
        )
        fig.update_layout(xaxis_title='Model', yaxis_title='Model')
        fig.show()
else:
    print('No RA pairwise level summary available.')

RA pairwise disagreement by level (highest mean difference):


,Affinity_Level,Model_1,Model_2,mean_diff,median_diff,std_diff,n_pairs
5,High,gpt-5.4-nano,ministral-3-14b-cloud,27.184615,25.0,14.477237,65
3,High,gpt-5.4-mini,gpt-5.4-nano,25.053333,25.0,15.846300,75
1,High,gemma4-31b-cloud,gpt-5.4-nano,24.947368,25.0,13.169556,76
0,High,gemma4-31b-cloud,gpt-5.4-mini,8.461538,7.0,7.208747,78
4,High,gpt-5.4-mini,ministral-3-14b-cloud,7.779412,7.0,6.810540,68
2,High,gemma4-31b-cloud,ministral-3-14b-cloud,6.086957,5.0,6.293646,69
11,Low,gpt-5.4-nano,ministral-3-14b-cloud,26.863112,25.0,18.459824,3207
10,Low,gpt-5.4-mini,ministral-3-14b-cloud,23.958874,20.0,17.067240,3623
8,Low,gemma4-31b-cloud,ministral-3-14b-cloud,21.655853,20.0,16.324545,3699
7,Low,gemma4-31b-cloud,gpt-5.4-nano,11.134465,0.0,18.173734,3592


In [81]:
# Compact interpretation table
interpretation_summary = pd.DataFrame(
    [
        {
            'Dataset': 'PRP',
            'Mean_CV': prp_unit_coherency_lvl['cv'].mean(),
            'Median_CV': prp_unit_coherency_lvl['cv'].median(),
            'Mean_MAD': prp_unit_coherency_lvl['mad_affinity'].mean(),
            'Median_MAD': prp_unit_coherency_lvl['mad_affinity'].median(),
            'Pct_Low_Mean_Flag': 100.0 * prp_unit_coherency_lvl['cv_low_mean_flag'].mean(),
            'N_Units': len(prp_unit_coherency_lvl),
        },
        {
            'Dataset': 'RA',
            'Mean_CV': ra_unit_coherency_lvl['cv'].mean(),
            'Median_CV': ra_unit_coherency_lvl['cv'].median(),
            'Mean_MAD': ra_unit_coherency_lvl['mad_affinity'].mean(),
            'Median_MAD': ra_unit_coherency_lvl['mad_affinity'].median(),
            'Pct_Low_Mean_Flag': 100.0 * ra_unit_coherency_lvl['cv_low_mean_flag'].mean(),
            'N_Units': len(ra_unit_coherency_lvl),
        },
    ]
)

print('Global coherency summary (CV + MAD):')
display(interpretation_summary)

print('Interpretation notes:')
print('- Low CV + Low MAD: strong cross-model agreement')
print('- High CV + Low mean affinity: inspect MAD before concluding disagreement')
print('- High CV + High MAD: substantive disagreement across models')

Global coherency summary (CV + MAD):


,Dataset,Mean_CV,Median_CV,Mean_MAD,Median_MAD,Pct_Low_Mean_Flag,N_Units
0,PRP,0.529636,0.525210,9.652933,8.160,1.333333,600
1,RA,1.208172,1.276569,10.974695,10.625,27.916667,4800


Interpretation notes:
- Low CV + Low MAD: strong cross-model agreement
- High CV + Low mean affinity: inspect MAD before concluding disagreement
- High CV + High MAD: substantive disagreement across models
